In [40]:
import tensorflow as tf
import keras
from keras_preprocessing.image import ImageDataGenerator
datagen=ImageDataGenerator(validation_split=0.25)

In [47]:
train_generator=datagen.flow_from_directory(
directory="Animal/",
batch_size=32,
seed=42,
subset="training",
color_mode="rgb",
shuffle=True,
class_mode="categorical",
target_size=(256,256))

Found 21749 images belonging to 27 classes.


In [48]:
valid_generator=datagen.flow_from_directory(
directory="Animal/",
batch_size=32,
subset="validation",
seed=42,
shuffle=True,
color_mode="rgb",
class_mode="categorical",
target_size=(256,256))

Found 7238 images belonging to 27 classes.


In [45]:
from keras.applications.inception_v3 import InceptionV3
from keras.preprocessing import image
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D
from keras import backend as K

# create the base pre-trained model
base_model = InceptionV3(weights='imagenet', include_top=False)

# add a global spatial average pooling layer
x = base_model.output
x = GlobalAveragePooling2D()(x)
# let's add a fully-connected layer
x = Dense(1024, activation='relu')(x)
# and a logistic layer -- let's say we have 30 classes
predictions = Dense(27, activation='softmax')(x)

# this is the model we will train
model = Model(inputs=base_model.input, outputs=predictions)

# first: train only the top layers (which were randomly initialized)
# i.e. freeze all convolutional InceptionV3 layers
for layer in base_model.layers:
    layer.trainable = False

# compile the model (should be done *after* setting layers to non-trainable)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            (None, None, None, 3 0                                            
__________________________________________________________________________________________________
conv2d_95 (Conv2D)              (None, None, None, 3 864         input_2[0][0]                    
__________________________________________________________________________________________________
batch_normalization_95 (BatchNo (None, None, None, 3 96          conv2d_95[0][0]                  
__________________________________________________________________________________________________
activation_95 (Activation)      (None, None, None, 3 0           batch_normalization_95[0][0]     
____________________________________________________________________________________________

In [49]:
model.fit_generator(
        train_generator,
        steps_per_epoch=170,
        validation_data=valid_generator,
        validation_steps=56,
        epochs=5)

Epoch 1/5
170/170 [==============================] - 645s 4s/step - loss: 0.9692 - accuracy: 0.7491 - val_loss: 58.2493 - val_accuracy: 0.1780
Epoch 2/5
  3/170 [..............................] - ETA: 7:51 - loss: 0.6267 - accuracy: 0.8125

KeyboardInterrupt: 

In [50]:
# at this point, the top layers are well trained and we can start fine-tuning
# convolutional layers from inception V3. We will freeze the bottom N layers
# and train the remaining top layers.

# let's visualize layer names and layer indices to see how many layers
# we should freeze:
for i, layer in enumerate(base_model.layers):
   print(i, layer.name)

# we chose to train the top 2 inception blocks, i.e. we will freeze
# the first 249 layers and unfreeze the rest:
for layer in model.layers[:249]:
   layer.trainable = False
for layer in model.layers[249:]:
   layer.trainable = True

# we need to recompile the model for these modifications to take effect
# we use SGD with a low learning rate
from keras.optimizers import SGD
model.compile(optimizer=SGD(lr=0.0001, momentum=0.9), loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

0 input_2
1 conv2d_95
2 batch_normalization_95
3 activation_95
4 conv2d_96
5 batch_normalization_96
6 activation_96
7 conv2d_97
8 batch_normalization_97
9 activation_97
10 max_pooling2d_5
11 conv2d_98
12 batch_normalization_98
13 activation_98
14 conv2d_99
15 batch_normalization_99
16 activation_99
17 max_pooling2d_6
18 conv2d_103
19 batch_normalization_103
20 activation_103
21 conv2d_101
22 conv2d_104
23 batch_normalization_101
24 batch_normalization_104
25 activation_101
26 activation_104
27 average_pooling2d_10
28 conv2d_100
29 conv2d_102
30 conv2d_105
31 conv2d_106
32 batch_normalization_100
33 batch_normalization_102
34 batch_normalization_105
35 batch_normalization_106
36 activation_100
37 activation_102
38 activation_105
39 activation_106
40 mixed0
41 conv2d_110
42 batch_normalization_110
43 activation_110
44 conv2d_108
45 conv2d_111
46 batch_normalization_108
47 batch_normalization_111
48 activation_108
49 activation_111
50 average_pooling2d_11
51 conv2d_107
52 conv2d_109
53 co

conv2d_186 (Conv2D)             (None, None, None, 3 442368      activation_185[0][0]             
__________________________________________________________________________________________________
conv2d_187 (Conv2D)             (None, None, None, 3 442368      activation_185[0][0]             
__________________________________________________________________________________________________
average_pooling2d_18 (AveragePo (None, None, None, 2 0           mixed9[0][0]                     
__________________________________________________________________________________________________
conv2d_180 (Conv2D)             (None, None, None, 3 655360      mixed9[0][0]                     
__________________________________________________________________________________________________
batch_normalization_182 (BatchN (None, None, None, 3 1152        conv2d_182[0][0]                 
__________________________________________________________________________________________________
batch_norm

In [ ]:
model.fit_generator(
        train_generator,
        steps_per_epoch=170,
        validation_data=valid_generator,
        validation_steps=56,
        shuffle=True,
        epochs=1)

Epoch 1/1
154/170 [==========================>...] - ETA: 57s - loss: 0.3944 - accuracy: 0.8883 

/home/habeeb-mikro/anaconda3/lib/python3.7/site-packages/PIL/TiffImagePlugin.py:802: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


169/170 [============================>.] - ETA: 3s - loss: 0.3887 - accuracy: 0.8899

In [8]:
model.save('Animals.h5')

In [9]:
model.evaluate_generator(generator=valid_generator,
steps=50)

[0.2971894145011902, 0.9549999833106995]

In [10]:
labels = (train_generator.class_indices)
labels

{'Bear': 0,
 'Butterfly': 1,
 'California  bighorn sheep': 2,
 'Cat': 3,
 'Cow': 4,
 'Deer': 5,
 'Dog': 6,
 'Elephant': 7,
 'Fox': 8,
 'Gorilla': 9,
 'Horse': 10,
 'Jaguar': 11,
 'Lion': 12,
 'Monkey': 13,
 'Orangutan': 14,
 'Pangolin': 15,
 'Panther': 16,
 'Rhinoceros': 17,
 'Saola': 18,
 'Sheep': 19,
 'Spider': 20,
 'Squirrel': 21,
 'Tiger': 22,
 'Wolf': 23,
 'Zebra': 24,
 'hen': 25,
 'leopard': 26}

In [20]:
labels = dict((v,k) for k,v in labels.items())

In [21]:
labels

{0: 'Bear',
 1: 'Butterfly',
 2: 'California  bighorn sheep',
 3: 'Cat',
 4: 'Cow',
 5: 'Deer',
 6: 'Dog',
 7: 'Elephant',
 8: 'Fox',
 9: 'Gorilla',
 10: 'Horse',
 11: 'Jaguar',
 12: 'Lion',
 13: 'Monkey',
 14: 'Orangutan',
 15: 'Pangolin',
 16: 'Panther',
 17: 'Rhinoceros',
 18: 'Saola',
 19: 'Sheep',
 20: 'Spider',
 21: 'Squirrel',
 22: 'Tiger',
 23: 'Wolf',
 24: 'Zebra',
 25: 'hen',
 26: 'leopard'}

In [39]:
import cv2
import numpy as np
train_generator.reset()
img = cv2.imread('tiger.jpg')

img = cv2.resize(img,(256,256))

img = np.reshape(img,[1,256,256,3])

classes = model.predict_classes(img)

AttributeError: 'Model' object has no attribute 'predict_classes'

0.8779592

In [35]:
predictions = (np.argmax(classes, axis=1))

In [36]:
pred = [labels[k] for k in predictions]

In [37]:
pred

['Sheep']